# NeuroVLM Quantitative Metrics — v2 (Semantic Metrics)

Evaluates both generative directions across three datasets.
BLEU / ROUGE have been replaced with meaning-aware metrics (see `why_not_bleu_rouge.md`).

| Direction | Datasets | Modes | Metrics |
|---|---|---|---|
| **Brain → Text** | Networks, PubMed, NeuroVault | Short & Long | **NeuroVLM Latent Sim** *(domain-specific)*, BERTScore F1, Semantic Cosine Sim, cleaned-reference variants, cleaned-minus-raw deltas |
| **Text → Brain** | Networks, PubMed, NeuroVault | — | Pearson *r*, Spearman ρ, Dice (pct-90), BrainSpace spin-test p-value |

**NeuroVLM Latent Similarity** encodes the generated text through NeuroVLM's own SPECTER
encoder + InfoNCE projection head, then measures cosine similarity to the original brain
embedding in the shared latent space — the most model-grounded evaluation available.

```
Prerequisites
pip install neurovlm[metrics,llm]
pip install bert-score sentence-transformers nltk scipy
# Optional, required for exact NCT-style MNI→fsaverage6 spin tests:
pip install neuromaps brainspace
# Ollama (fast path):
ollama pull qwen2.5:3b-instruct
```


In [14]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import hashlib
import json
import re
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from neurovlm import NeuroVLM
from neurovlm.data import load_dataset, load_latent, load_masker
from neurovlm.metrics import (
    pearson_correlation,
    dice_percentile,
    nct_dice_spin_test_nifti,
    nct_dice_spin_test_surface,
    mni152_to_fsaverage_arrays,
)

# Semantic text metrics
from bert_score import score as bert_score
from sentence_transformers import SentenceTransformer, util as st_util

# Spatial metrics
from scipy.stats import spearmanr


In [15]:
# ── LLM settings ──────────────────────────────────────────────────────────────
LLM_BACKEND = "ollama"               # "ollama" | "huggingface"
LLM_MODEL   = "qwen2.5:3b-instruct"  # Ollama: any pulled model

# ── BERTScore model ───────────────────────────────────────────────────────────
# Best for scientific text; swap for "distilbert-base-uncased" if OOM.
BERTSCORE_MODEL = "microsoft/deberta-xlarge-mnli"

# ── Evaluation budget ─────────────────────────────────────────────────────────
MAX_B2T = 30   # samples per dataset for brain-to-text
MAX_T2B = 50   # samples per dataset for text-to-brain

# ── Retrieval settings ────────────────────────────────────────────────────────
B2T_TOP_K   = 5     # context rows passed to LLM
B2T_SIM_THR = 0.35  # cosine-similarity floor; relaxed if no rows pass
B2T_DATASETS = ["kg_mesh", "cogatlas"]  # retrieval corpora for brain-to-text

# ── Abstract cleaning cache / QC ──────────────────────────────────────────────
CLEAN_GT_SIM_THRESHOLD = 0.85
CLEAN_GT_SPOTCHECK_N = 5
CLEAN_ABSTRACT_PROMPT_VERSION = "v3_no_methods_or_participants"
_EVAL_DIR = Path("docs/03_evaluation") if Path("docs/03_evaluation").exists() else Path(".")
CLEAN_ABSTRACT_CACHE_PATH = _EVAL_DIR / "cleaned_b2t_abstracts_cache.json"

# ── Spatial metric settings ───────────────────────────────────────────────────
DICE_PCT = 90
DICE_SENSITIVITY_PCTS = [80, 85, 90, 95]
SPIN_TEST_N_PERM = 1000
SPIN_TEST_RANDOM_STATE = 13

# NCT-style spin test settings. fsaverage6 corresponds to neuromaps/FreeSurfer
# density "41k". neuromaps wraps the Wu et al. (2018) nonlinear MNI→fsaverage
# registrations via transforms.mni152_to_fsaverage; BrainSpace handles rotations.
SPIN_USE_NEUROMAPS = True
SPIN_FSAVERAGE_DENSITY = "41k"
SPIN_TRANSFORM_METHOD = "linear"


In [16]:
nvlm   = NeuroVLM()
masker = load_masker()

print("Loading sentence transformer...")
_st_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Ready.")

Loading sentence transformer...
Ready.


---
## SPECTER2 Adapter Check

`NeuroVLM._encode_text()` lazy-loads `load_model("specter")`, which resolves to `Specter()`.
In the current codebase, `Specter()` defaults to `model="allenai/specter2_aug2023refresh"` and `adapter="adhoc_query"`; it does **not** default to the proximity adapter (`allenai/specter2`). The InfoNCE projection head was trained downstream of the cached `latent_specter2_adhoc.pt` embeddings, so changing adapters would require at least a distribution sanity check and probably retraining if cosine agreement is low.


In [17]:
from neurovlm.models import Specter

print("NeuroVLM _encode_text() uses load_model('specter') -> Specter().")
print("Current Specter default: model='allenai/specter2_aug2023refresh', adapter='adhoc_query'.")
print("Projection-head training artifacts reference latent_specter2_adhoc.pt, so the head is aligned to adhoc-query embeddings.")


def specter_adapter_cosine_sanity(texts, other_adapter="proximity"):
    """Compare current adhoc-query embeddings against another SPECTER2 adapter.

    High cosine agreement (>0.9) suggests the existing InfoNCE head may remain usable;
    lower agreement means the head is seeing a shifted embedding distribution and should
    be retrained before using that adapter for reported metrics.
    """
    base = Specter(adapter="adhoc_query", device=str(nvlm.device))
    other = Specter(adapter=other_adapter, device=str(nvlm.device))
    with torch.no_grad():
        z_base = F.normalize(base(texts).cpu(), dim=-1)
        z_other = F.normalize(other(texts).cpu(), dim=-1)
    sims = F.cosine_similarity(z_base, z_other).numpy()
    return pd.DataFrame({"text": texts, "adapter": other_adapter, "cosine_to_adhoc": sims})

# Example, intentionally not run automatically because it may download the other adapter:
# sample_texts = [d["long_gt"] for d in pubmed_data_b2t[:3]]
# specter_adapter_cosine_sanity(sample_texts, other_adapter="proximity")


NeuroVLM _encode_text() uses load_model('specter') -> Specter().
Current Specter default: model='allenai/specter2_aug2023refresh', adapter='adhoc_query'.
Projection-head training artifacts reference latent_specter2_adhoc.pt, so the head is aligned to adhoc-query embeddings.


---
## Dataset Preparation

In [18]:
# ── Networks dataset ──────────────────────────────────────────────────────────
SHORT_LABELS = {
    "Language":               "Language network supporting speech comprehension and production.",
    "Auditory":               "Auditory network for acoustic processing and speech encoding.",
    "Default Mode":           "Default mode network for self-referential thought and memory retrieval.",
    "Frontoparietal Control": "Frontoparietal control network for executive function and working memory.",
    "Attention":              "Dorsal attention network for goal-directed visuospatial attention.",
    "Visual":                 "Visual network for feature processing and object recognition.",
    "Motor":                  "Sensorimotor network for movement planning, initiation, and execution.",
    "Cingulo-Opercular":      "Salience network detecting salient events and driving performance monitoring.",
}

LONG_LABELS = {
    "Language": """Language network (LAN; perisylvian language network; frontotemporal language system).Primary regions: left inferior frontal gyrus (Broca's complex), posterior 
                 superior/middle temporal gyrus, temporoparietal junction/angular gyrus. Function: speech/text comprehension and production—semantics, syntax, phonology, and 
                 sentence-level integration.""",

    "Auditory": """ Auditory network (AUD; auditory cortex network). Primary regions: Heschl's gyrus (primary auditory cortex), planum temporale, superior temporal gyrus. Function: acoustic 
        feature analysis (pitch/timbre/timing), auditory scene analysis, early speech-sound encoding, and detection of salient sounds.""",

    "Default Mode": """ Default mode network (DMN; default network; default state network). Primary regions: medial prefrontal cortex, posterior cingulate/precuneus, angular gyrus (lateral parietal).
        Function: internally oriented cognition—autobiographical/episodic memory, self-referential thought, future simulation, social inference, and narrative/semantic integration.""",

    "Frontoparietal Control": """Frontoparietal control network (FPCN; frontoparietal network/FPN; central executive network/CEN). Primary regions: dorsolateral/rostrolateral prefrontal cortex 
        (middle frontal), posterior parietal cortex (inferior parietal/around intraparietal sulcus). Function: goal maintenance and flexible executive control—working memory, 
        rule implementation, planning, and rapid task switching.""",

    "Attention": """ Dorsal attention network. Primary regions: intraparietal sulcus/superior parietal lobule and frontal eye fields. Function: goal-directed, voluntary control of visuospatial 
        attention.""",

    "Visual": """Visual network (VIS; occipital visual network). Primary regions: calcarine cortex/V1 (cuneus/lingual), extrastriate occipital cortex, ventral occipitotemporal visual 
    areas. Function: visual feature processing (form/color/motion), object/scene representations, and visuospatial analysis.""",

    "Motor": """Motor network (motor/sensorimotor network; SMN). Primary regions: primary motor cortex (precentral gyrus), supplementary motor area, premotor cortex; with primary 
    somatosensory cortex (postcentral gyrus) for sensorimotor integration. Function: movement planning, initiation, and execution, plus proprioceptive/tactile feedback used to control 
    actions and maintain body-state representations.""",

    "Cingulo-Opercular": """Salience network (SN; cingulo-opercular network/CON; midcingulo-insular network). Primary regions: anterior insula/frontal operculum, dorsal anterior cingulate/medial frontal cortex. 
    Function: detects behaviorally relevant internal/external events (including interoceptive/affective signals), prioritizes attention and arousal, and drives performance monitoring 
    and control adjustments (error/conflict/uncertainty).""",
}


def _normalize_expected_text(text: str) -> str:
    """Collapse indentation/newlines from triple-quoted expected text."""
    return re.sub(r"\s+", " ", str(text or "")).strip()

_LABEL_TO_DU = {
    "Language":               "LANG",
    "Auditory":               "AUD",
    "Default Mode":           "DN-A",
    "Frontoparietal Control": "FPN-A",
    "Attention":              "dATN-A",
    "Visual":                 "VIS-C",
    "Motor":                  "SMOT-A",
    "Cingulo-Opercular":      "CG-OP",
}

all_net_latents = load_latent("networks_neuro")
du = all_net_latents["Du"]

networks_data = {
    label: {
        "latent":   du[du_key],
        "short_gt": _normalize_expected_text(SHORT_LABELS[label]),
        "long_gt":  _normalize_expected_text(LONG_LABELS[label]),
    }
    for label, du_key in _LABEL_TO_DU.items()
    if du_key in du
}

print(f"Networks loaded: {list(networks_data.keys())}")

Networks loaded: ['Language', 'Auditory', 'Default Mode', 'Frontoparietal Control', 'Attention', 'Visual', 'Motor', 'Cingulo-Opercular']


In [19]:
# ── PubMed dataset ────────────────────────────────────────────────────────────
df_pubs = load_dataset("pubmed_text")
print("PubMed columns:", df_pubs.columns.tolist())
print(f"Total rows: {len(df_pubs)}")

if "split" in df_pubs.columns:
    df_test = df_pubs[df_pubs["split"] == "test"].reset_index(drop=True)
    print(f"Test split rows: {len(df_test)}")
else:
    print("WARNING: no 'split' column found — using full dataset as test proxy.")
    df_test = df_pubs.copy().reset_index(drop=True)

_pmid_col     = "pmid"        if "pmid"        in df_test.columns else df_test.columns[0]
_title_col    = "name"        if "name"        in df_test.columns else "title"
_abstract_col = "description" if "description" in df_test.columns else "abstract"

pubmed_latents, pubmed_pmids = load_latent("pubmed_images")
pubmed_pmids = np.asarray(pubmed_pmids)

mask = np.isin(pubmed_pmids, df_test[_pmid_col].values)
aligned_latents = pubmed_latents[mask]
aligned_pmids   = pubmed_pmids[mask]

pmid_to_row = df_test.set_index(_pmid_col)

pubmed_data = []
for i, pmid in enumerate(aligned_pmids):
    if pmid not in pmid_to_row.index:
        continue
    row      = pmid_to_row.loc[pmid]
    title    = str(row[_title_col])    if _title_col    in row.index else ""
    abstract = str(row[_abstract_col]) if _abstract_col in row.index else ""
    pubmed_data.append({
        "pmid":     pmid,
        "latent":   aligned_latents[i],
        "short_gt": title,
        "long_gt":  abstract,
    })

pubmed_data_b2t = pubmed_data[:MAX_B2T] if MAX_B2T else pubmed_data
pubmed_data_t2b = pubmed_data[:MAX_T2B] if MAX_T2B else pubmed_data

print(f"PubMed B2T samples: {len(pubmed_data_b2t)}")
print(f"PubMed T2B samples: {len(pubmed_data_t2b)}")

PubMed columns: ['pmid', 'pmcid', 'doi', 'name', 'description', 'train', 'test', 'val']
Total rows: 30826
PubMed B2T samples: 30
PubMed T2B samples: 50


In [20]:
# ── NeuroVault dataset ────────────────────────────────────────────────────────
df_nv      = load_dataset("neurovault_text")
df_nv_meta = load_dataset("neurovault_images_meta")
nv_latents = load_latent("neurovault_images")

print("NeuroVault pub columns:",  df_nv.columns.tolist())
print("NeuroVault meta columns:", df_nv_meta.columns.tolist())
print(f"NV latents shape: {nv_latents.shape}")

_doi_pub  = "doi"      if "doi"      in df_nv.columns      else df_nv.columns[0]
_doi_meta = "doi"      if "doi"      in df_nv_meta.columns else df_nv_meta.columns[0]
_title_nv = "title"    if "title"    in df_nv.columns      else df_nv.columns[1]
_abs_nv   = "abstract" if "abstract" in df_nv.columns      else df_nv.columns[2]

neurovault_data = []
for _, pub_row in df_nv.iterrows():
    doi      = pub_row[_doi_pub]
    title    = str(pub_row[_title_nv])
    abstract = str(pub_row[_abs_nv])

    img_mask    = df_nv_meta[_doi_meta] == doi
    img_indices = np.where(img_mask.values)[0]

    if len(img_indices) == 0 or img_indices[0] >= len(nv_latents):
        continue

    img_idx = int(img_indices[0])
    neurovault_data.append({
        "doi":      doi,
        "latent":   nv_latents[img_idx],
        "short_gt": title,
        "long_gt":  abstract,
    })

neurovault_data_b2t = neurovault_data[:MAX_B2T] if MAX_B2T else neurovault_data
neurovault_data_t2b = neurovault_data[:MAX_T2B] if MAX_T2B else neurovault_data

print(f"NeuroVault B2T samples: {len(neurovault_data_b2t)}")
print(f"NeuroVault T2B samples: {len(neurovault_data_t2b)}")

NeuroVault pub columns: ['doi', 'title', 'abstract']
NeuroVault meta columns: ['Unnamed: 0', 'id', 'collection_id', 'contrast_definition', 'name', 'doi']
NV latents shape: torch.Size([3183, 384])
NeuroVault B2T samples: 30
NeuroVault T2B samples: 50


In [21]:
from neurovlm.llm_summary import _generate_with_ollama, _generate_with_huggingface

_CLEAN_ABSTRACT_SYSTEM = """You convert neuroimaging abstracts into concise finding-only reference text for fair semantic evaluation.

Hard rules:
- Remove all participant/sample language. Do not mention volunteers, patients, subjects, controls, sample size, age, sex, handedness, or demographics.
- Remove all acquisition/procedure details. Do not mention PET, fMRI, MRI, scanner field strength, radiotracers, injections, image acquisition, preprocessing, thresholds, software, or statistical models.
- Remove all numeric result details and inferential statistics. Do not mention p-values, t/F/r values, cluster sizes, coordinates, percentages, timing, doses, or thresholds.
- Remove generic study framing such as "we measured", "we investigated", "the study used", or "activation was measured".
- Preserve only the semantic neuroscience content: brain regions, direction of activation/connectivity if stated, cognitive functions, stimuli/tasks/contrasts in plain language, disorders/conditions, and core findings.
- Do not add facts. Do not preserve unsupported methodological facts just because they are in the original.
- Return 1-4 plain sentences and nothing else.

Example:
Original: Cerebral activation was measured with positron emission tomography in ten human volunteers. The primary auditory cortex showed increased activity in response to noise bursts, whereas acoustically matched speech syllables activated secondary auditory cortices bilaterally. Instructions to make judgments about syllables modulated frontal activity.
Cleaned: Noise bursts preferentially engaged primary auditory cortex, while matched speech syllables engaged bilateral secondary auditory cortices. Judgments about syllables modulated frontal activity."""


def _load_clean_abstract_cache() -> dict:
    if CLEAN_ABSTRACT_CACHE_PATH.exists():
        with CLEAN_ABSTRACT_CACHE_PATH.open("r", encoding="utf-8") as f:
            return json.load(f)
    return {}


def _save_clean_abstract_cache(cache: dict) -> None:
    with CLEAN_ABSTRACT_CACHE_PATH.open("w", encoding="utf-8") as f:
        json.dump(cache, f, indent=2, ensure_ascii=False, sort_keys=True)


def _clean_cache_key(dataset: str, sample_id: str, text: str) -> str:
    digest = hashlib.sha256(str(text or "").encode("utf-8")).hexdigest()[:16]
    return f"{CLEAN_ABSTRACT_PROMPT_VERSION}|{LLM_BACKEND}|{LLM_MODEL}|{dataset}|{sample_id}|{digest}"


def clean_abstract_for_eval(text: str) -> str:
    """Rewrite a scientific abstract into a finding-focused reference for B2T metrics."""
    text = str(text or "").strip()
    if not text or text.lower() == "nan":
        return ""

    messages = [
        {"role": "system", "content": _CLEAN_ABSTRACT_SYSTEM},
        {"role": "user", "content": f"Clean this abstract. Return only the cleaned finding-focused text:\n\n{text}"},
    ]
    if LLM_BACKEND == "ollama":
        return _generate_with_ollama(messages, model_name=LLM_MODEL, verbose=False).strip()
    if LLM_BACKEND == "huggingface":
        return _generate_with_huggingface(messages, model_name=LLM_MODEL, verbose=False).strip()
    raise ValueError(f"Unknown LLM_BACKEND: {LLM_BACKEND}")


def _clean_gt_semantic_sim(original: str, cleaned: str) -> float:
    if not str(original or "").strip() or not str(cleaned or "").strip():
        return np.nan
    emb1 = _st_model.encode(str(original), convert_to_tensor=True)
    emb2 = _st_model.encode(str(cleaned), convert_to_tensor=True)
    return float(st_util.cos_sim(emb1, emb2))


def add_clean_b2t_references(records, dataset, id_key, desc):
    """Attach cleaned references, using a persistent JSON cache to avoid reruns."""
    cache = _load_clean_abstract_cache()
    cache_changed = False
    for d in tqdm(records, desc=desc):
        sample_id = str(d.get(id_key, ""))
        key = _clean_cache_key(dataset, sample_id, d["long_gt"])
        cached = cache.get(key)
        if cached is None:
            cleaned = clean_abstract_for_eval(d["long_gt"])
            cache[key] = {
                "dataset": dataset,
                "sample_id": sample_id,
                "prompt_version": CLEAN_ABSTRACT_PROMPT_VERSION,
                "backend": LLM_BACKEND,
                "model": LLM_MODEL,
                "original": d["long_gt"],
                "cleaned": cleaned,
            }
            cache_changed = True
        else:
            cleaned = cached["cleaned"]

        d["short_gt_clean"] = d["short_gt"]
        d["long_gt_clean"] = cleaned
        d["long_gt_clean_sim"] = _clean_gt_semantic_sim(d["long_gt"], d["long_gt_clean"])
        d["long_gt_clean_flag"] = bool(
            np.isfinite(d["long_gt_clean_sim"]) and d["long_gt_clean_sim"] < CLEAN_GT_SIM_THRESHOLD
        )

    if cache_changed:
        _save_clean_abstract_cache(cache)
    return records

# Networks labels are already clean; only PubMed and NeuroVault abstracts are rewritten.
pubmed_data_b2t = add_clean_b2t_references(pubmed_data_b2t, "pubmed", "pmid", "Cleaning PubMed abstracts")
neurovault_data_b2t = add_clean_b2t_references(neurovault_data_b2t, "neurovault", "doi", "Cleaning NeuroVault abstracts")

_clean_qc_rows = []
for dataset, records in [("pubmed", pubmed_data_b2t), ("neurovault", neurovault_data_b2t)]:
    for i, d in enumerate(records):
        _clean_qc_rows.append({
            "dataset": dataset,
            "sample": str(d.get("pmid", d.get("doi", i))),
            "original": d["long_gt"],
            "cleaned": d["long_gt_clean"],
            "orig_clean_sem_sim": d["long_gt_clean_sim"],
            "needs_manual_review": d["long_gt_clean_flag"],
        })

clean_abstract_qc_df = pd.DataFrame(_clean_qc_rows)
print(f"Cleaned references ready from cache: {CLEAN_ABSTRACT_CACHE_PATH}")
print(f"Manual-review flags: {clean_abstract_qc_df['needs_manual_review'].sum()} / {len(clean_abstract_qc_df)}")
display(clean_abstract_qc_df[["dataset", "sample", "orig_clean_sem_sim", "needs_manual_review"]].round(3))


Cleaning PubMed abstracts:   0%|          | 0/30 [00:00<?, ?it/s]

Cleaning NeuroVault abstracts:   0%|          | 0/30 [00:00<?, ?it/s]

Cleaned references ready from cache: cleaned_b2t_abstracts_cache.json
Manual-review flags: 49 / 60


,dataset,sample,orig_clean_sem_sim,needs_manual_review
0,pubmed,1589767,0.908,False
1,pubmed,8530552,0.702,True
2,pubmed,8624678,0.799,True
3,pubmed,8670634,0.797,True
4,pubmed,8994101,0.689,True
5,pubmed,9038284,0.730,True
6,pubmed,9065511,0.820,True
7,pubmed,9084599,0.764,True
8,pubmed,9106283,0.720,True
9,pubmed,9121583,0.806,True


In [22]:
# Spot-check a few cleaned abstracts before trusting cleaned-reference metrics.
# Good rewrites should keep semantic cosine similarity above CLEAN_GT_SIM_THRESHOLD.
spotcheck_clean_abstracts = (
    clean_abstract_qc_df
    .sort_values(["needs_manual_review", "orig_clean_sem_sim"], ascending=[False, True])
    .head(CLEAN_GT_SPOTCHECK_N)
)

for _, row in spotcheck_clean_abstracts.iterrows():
    print("=" * 80)
    print(f"{row['dataset']} | {row['sample']} | sim={row['orig_clean_sem_sim']:.3f} | review={row['needs_manual_review']}")
    print("\nORIGINAL:")
    print(str(row["original"])[:1200])
    print("\nCLEANED:")
    print(str(row["cleaned"])[:1200])


pubmed | 9391021 | sim=0.644 | review=True

ORIGINAL:
To obtain a better understanding of the cortical representation of bimanual coordination, we measured regional cerebral blood flow (rCBF) with 15O-labeled water and positron emission tomography (PET). To detect areas with changes of rCBF during bimanual finger movements of different characteristics, we studied 12 right-handed normal volunteers. A complete session consisted of three rest scans and six scans with acoustically paced (1 Hz) bimanual, mirror, or parallel sequential finger movements. Activation of the right dorsal premotor area (PMd) extending to the posterior supplementary motor area (SMA) was significantly stronger during the parallel movements than during the mirror sequential movements (p < 0.05, at cluster level with correction for multiple comparisons). To determine whether these cortical areas truly represented bimanual coordination, a different group of nine normal volunteers was studied with a different task. Sub

---
## Part A — Brain-to-Text (BERTScore, Semantic Sim, METEOR, NeuroVLM Latent Sim)

For each brain image we:
1. Retrieve the most similar text entries via contrastive similarity.
2. Generate with the LLM twice — once in **short** mode, once in **long** mode.
3. Compute BERTScore, sentence-level cosine similarity, and NeuroVLM latent similarity.

**NeuroVLM Latent Similarity** is the most meaningful metric here:
it encodes the generated text through NeuroVLM's own SPECTER + projection head
and computes cosine similarity against the original brain image embedding
in the shared latent space. A paraphrase that preserves meaning will score
as highly as a literal match — and the metric is grounded in the model's own
representation, not an external embedding space.

See `why_not_bleu_rouge.md` for the rationale for dropping BLEU / ROUGE.

In [23]:
import traceback


def _semantic_sim(gen: str, gt: str) -> float:
    emb1 = _st_model.encode(gen, convert_to_tensor=True)
    emb2 = _st_model.encode(gt,  convert_to_tensor=True)
    return float(st_util.cos_sim(emb1, emb2))


def _bertscore_single(generated: str, reference: str):
    P, R, F1 = bert_score(
        cands=[generated],
        refs=[reference],
        lang="en",
        model_type=BERTSCORE_MODEL,
        verbose=False,
    )
    return float(P[0]), float(R[0]), float(F1[0])


def _nvlm_latent_sim(brain_query_emb: torch.Tensor, generated: str) -> float:
    """Cosine similarity in NeuroVLM's shared latent space."""
    nvlm._ensure_projection_heads()
    with torch.no_grad():
        raw_emb = nvlm._encode_text(generated)                         # (1, 768)
        z_text  = nvlm._proj_head_text_infonce(raw_emb.to(nvlm.device)) # project
        z_text  = F.normalize(z_text, dim=-1).cpu()                    # normalise
    z_brain = brain_query_emb.cpu()
    if z_brain.dim() == 1:
        z_brain = z_brain.unsqueeze(0)
    return float(F.cosine_similarity(z_brain, z_text))


def _b2t_once(table, user_prompt, gt_text, max_tokens, brain_query_emb, gt_text_clean=None):
    """Generate one B2T response and compute semantic metrics against raw and cleaned refs."""
    generated = nvlm.generate_llm_response(
        backend=LLM_BACKEND,
        model_name=LLM_MODEL,
        table=table,
        user_prompt=user_prompt,
        max_new_tokens=max_tokens,
        verbose=False,
    )

    bert_p, bert_r, bert_f1 = _bertscore_single(generated, gt_text)
    sem_sim = _semantic_sim(generated, gt_text)
    nvlm_sim = _nvlm_latent_sim(brain_query_emb, generated)

    rec = {
        "generated":  generated,
        "gt_text":    gt_text,
        "gt_text_clean": gt_text_clean if gt_text_clean is not None else gt_text,
        "bert_p":     bert_p,
        "bert_r":     bert_r,
        "bert_f1":    bert_f1,
        "sem_sim":    sem_sim,
        "nvlm_sim":   nvlm_sim,
    }

    if gt_text_clean is not None and gt_text_clean != gt_text:
        clean_p, clean_r, clean_f1 = _bertscore_single(generated, gt_text_clean)
        rec.update({
            "bert_p_clean":  clean_p,
            "bert_r_clean":  clean_r,
            "bert_f1_clean": clean_f1,
            "sem_sim_clean": _semantic_sim(generated, gt_text_clean),
        })
    else:
        rec.update({
            "bert_p_clean":  bert_p,
            "bert_r_clean":  bert_r,
            "bert_f1_clean": bert_f1,
            "sem_sim_clean": sem_sim,
        })

    return rec


def _format_context_summary(table):
    lines = []
    for _, row in table.iterrows():
        ds  = row.get("dataset", "?")
        ttl = str(row.get("title", "")).strip()
        sim = row.get("cosine_similarity", float("nan"))
        lines.append(f"  [{ds}] (sim={sim:.3f}) {ttl}")
    return "\n".join(lines)


def run_b2t(name, latent, short_gt, long_gt,
            short_prompt, long_prompt="",
            short_tokens=64, long_tokens=512,
            short_gt_clean=None, long_gt_clean=None):
    try:
        result    = nvlm.brain(latent).to_text(datasets=B2T_DATASETS)
        all_table = result.top_k(B2T_TOP_K)  # top-k per dataset
        table     = all_table[all_table["cosine_similarity"] > B2T_SIM_THR]
        if table.empty:
            table = all_table
        if len(table) > B2T_TOP_K:
            table = table.nlargest(B2T_TOP_K, "cosine_similarity").reset_index(drop=True)

        context_summary = _format_context_summary(table)
        brain_query_emb = result.query_embeddings

        records = []
        for mode, prompt, gt, gt_clean, tokens in [
            ("short", short_prompt, short_gt, short_gt_clean, short_tokens),
            ("long",  long_prompt,  long_gt,  long_gt_clean,  long_tokens),
        ]:
            rec = _b2t_once(table, prompt, gt, tokens, brain_query_emb, gt_text_clean=gt_clean)
            rec["name"]            = name
            rec["mode"]            = mode
            rec["context_summary"] = context_summary
            records.append(rec)

        return records

    except Exception as e:
        print(f"\n[B2T error] {name}: {type(e).__name__}: {e}")
        traceback.print_exc()
        return []


def show_b2t_texts(df, max_gt_chars=300, max_gen_chars=300):
    sep = "-" * 72
    prev_name = None
    for _, row in df.iterrows():
        name  = row.get("name", "?")
        mode  = row.get("mode", "?")
        gen   = str(row.get("generated",       "")).strip()
        gt    = str(row.get("gt_text",         "")).strip()
        gt_c  = str(row.get("gt_text_clean",   "")).strip()
        ctx   = str(row.get("context_summary", "")).strip()
        bf1   = row.get("bert_f1",  float("nan"))
        bf1c  = row.get("bert_f1_clean", float("nan"))
        sim   = row.get("sem_sim",  float("nan"))
        simc  = row.get("sem_sim_clean", float("nan"))
        nvs   = row.get("nvlm_sim", float("nan"))

        print(sep)
        if name != prev_name and ctx:
            print(f"[{name}]  — retrieved context:")
            print(ctx)
            prev_name = name

        print(f"  mode={mode}   BERTScore-F1={bf1:.3f} / clean={bf1c:.3f}  "
              f"SemanticSim={sim:.3f} / clean={simc:.3f}  NeuroVLM-Sim={nvs:.3f}")
        print(f"  EXPECTED : {gt[:max_gt_chars]}{'…' if len(gt) > max_gt_chars else ''}")
        if gt_c and gt_c != gt:
            print(f"  CLEAN GT : {gt_c[:max_gt_chars]}{'…' if len(gt_c) > max_gt_chars else ''}")
        print(f"  GENERATED: {gen[:max_gen_chars]}{'…' if len(gen) > max_gen_chars else ''}")
    print(sep)


# ── Prompts ────────────────────────────────────────────────────────────────────
SHORT_PROMPT_GENERAL = (
    "Reply with a single concise sentence (10-20 words) naming the main "
    "cognitive function or brain network. Output only that sentence."
)

SHORT_PROMPT_PUBMED = (
    "Generate ONLY a paper title (6-12 words) for the neuroimaging study this "
    "brain activation pattern represents. Output the title only — no abstract, "
    "no explanation."
)

LONG_PROMPT = ""


### A.1 Networks

In [24]:
b2t_networks_records = []

for net_name, d in tqdm(networks_data.items(), desc="Networks B2T"):
    recs = run_b2t(
        name         = net_name,
        latent       = d["latent"],
        short_gt     = d["short_gt"],
        long_gt      = d["long_gt"],
        short_prompt = SHORT_PROMPT_GENERAL,
        long_prompt  = LONG_PROMPT,
    )
    b2t_networks_records.extend(recs)

print(f"\nRecords collected: {len(b2t_networks_records)}")

b2t_net_df = pd.DataFrame(b2t_networks_records)

if b2t_net_df.empty:
    print("No records — check the [B2T error] messages above.")
else:
    cols = ["name", "mode", "bert_f1", "bert_p", "bert_r", "sem_sim", "nvlm_sim"]
    display(b2t_net_df[cols].round(3))
    print()
    show_b2t_texts(b2t_net_df)

Networks B2T:   0%|          | 0/8 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.



Records collected: 16


,name,mode,bert_f1,bert_p,bert_r,sem_sim,nvlm_sim
0,Language,short,0.697,0.660,0.738,0.625,0.331
1,Language,long,0.606,0.571,0.646,0.788,0.393
2,Auditory,short,0.707,0.689,0.725,0.595,0.383
3,Auditory,long,0.542,0.492,0.603,0.620,0.463
4,Default Mode,short,0.726,0.691,0.766,0.646,0.363
5,Default Mode,long,0.550,0.495,0.620,0.826,0.362
6,Frontoparietal Control,short,0.711,0.702,0.719,0.559,0.362
7,Frontoparietal Control,long,0.511,0.493,0.530,0.559,0.353
8,Attention,short,0.736,0.722,0.751,0.729,0.298
9,Attention,long,0.562,0.490,0.659,0.742,0.382



------------------------------------------------------------------------
[Language]  — retrieved context:
[kg_mesh] (sim=0.426) language network
  [kg_mesh] (sim=0.413) language abilities
  [kg_mesh] (sim=0.408) expressive language
  [kg_mesh] (sim=0.402) superior temporal regions
  [kg_mesh] (sim=0.396) frontotemporal regions
  mode=short   BERTScore-F1=0.697 / clean=0.697  SemanticSim=0.625 / clean=0.625  NeuroVLM-Sim=0.331
  EXPECTED : Language network supporting speech comprehension and production.
  GENERATED: The primary cognitive function is linguistic communication enabled by the language network.
------------------------------------------------------------------------
  mode=long   BERTScore-F1=0.606 / clean=0.606  SemanticSim=0.788 / clean=0.788  NeuroVLM-Sim=0.393
  EXPECTED : Language network (LAN; perisylvian language network; frontotemporal language system).Primary regions: left inferior frontal gyrus (Broca's complex), posterior superior/middle temporal gyrus, temporopa

### A.2 PubMed

*Short* → LLM generates a title, evaluated against the real title.  
*Long* → LLM generates freely, evaluated against the abstract.

In [25]:
b2t_pubmed_records = []

for d in tqdm(pubmed_data_b2t, desc="PubMed B2T"):
    recs = run_b2t(
        name           = str(d["pmid"]),
        latent         = d["latent"],
        short_gt       = d["short_gt"],
        long_gt        = d["long_gt"],
        short_gt_clean = d.get("short_gt_clean"),
        long_gt_clean  = d.get("long_gt_clean"),
        short_prompt   = SHORT_PROMPT_PUBMED,
        long_prompt    = LONG_PROMPT,
        short_tokens   = 48,
        long_tokens    = 512,
    )
    b2t_pubmed_records.extend(recs)

print(f"\nRecords collected: {len(b2t_pubmed_records)}")

b2t_pubmed_df = pd.DataFrame(b2t_pubmed_records)

if b2t_pubmed_df.empty:
    print("No records — check the [B2T error] messages above.")
else:
    display(b2t_pubmed_df.groupby("mode")[["bert_f1", "bert_f1_clean", "sem_sim", "sem_sim_clean", "nvlm_sim"]].mean().round(3))
    print()
    show_b2t_texts(b2t_pubmed_df)


PubMed B2T:   0%|          | 0/30 [00:00<?, ?it/s]


Records collected: 60


,bert_f1,bert_f1_clean,sem_sim,sem_sim_clean,nvlm_sim
mode,,,,,
long,0.504,0.487,0.398,0.355,0.345
short,0.602,0.602,0.391,0.391,0.310



------------------------------------------------------------------------
[1589767]  — retrieved context:
[cogatlas] (sim=0.479) word pronunciation
  [kg_mesh] (sim=0.469) native language
  [cogatlas] (sim=0.467) language processing
  [kg_mesh] (sim=0.463) language dominance
  [kg_mesh] (sim=0.463) natural speech
  mode=short   BERTScore-F1=0.633 / clean=0.633  SemanticSim=0.671 / clean=0.671  NeuroVLM-Sim=0.355
  EXPECTED : Lateralization of phonetic and pitch discrimination in speech processing.
  GENERATED: Pronunciation in Language Processing: Native Acquisition and Dominant Hemisphere Function
------------------------------------------------------------------------
  mode=long   BERTScore-F1=0.515 / clean=0.522  SemanticSim=0.554 / clean=0.583  NeuroVLM-Sim=0.395
  EXPECTED : Cerebral activation was measured with positron emission tomography in ten human volunteers. The primary auditory cortex showed increased activity in response to noise bursts, whereas acoustically matched spee

### A.3 NeuroVault

In [ ]:
b2t_nv_records = []

for d in tqdm(neurovault_data_b2t, desc="NeuroVault B2T"):
    recs = run_b2t(
        name           = str(d["doi"]),
        latent         = d["latent"],
        short_gt       = d["short_gt"],
        long_gt        = d["long_gt"],
        short_gt_clean = d.get("short_gt_clean"),
        long_gt_clean  = d.get("long_gt_clean"),
        short_prompt   = SHORT_PROMPT_GENERAL,
        long_prompt    = LONG_PROMPT,
    )
    b2t_nv_records.extend(recs)

print(f"\nRecords collected: {len(b2t_nv_records)}")

b2t_nv_df = pd.DataFrame(b2t_nv_records)

if b2t_nv_df.empty:
    print("No records — check the [B2T error] messages above.")
else:
    display(b2t_nv_df.groupby("mode")[["bert_f1", "bert_f1_clean", "sem_sim", "sem_sim_clean", "nvlm_sim"]].mean().round(3))
    print()
    show_b2t_texts(b2t_nv_df)


### A.4 Brain-to-Text Summary

In [ ]:
b2t_net_df["dataset"]    = "networks"
b2t_pubmed_df["dataset"] = "pubmed"
b2t_nv_df["dataset"]     = "neurovault"

b2t_all = pd.concat([b2t_net_df, b2t_pubmed_df, b2t_nv_df], ignore_index=True)

summary_b2t = (
    b2t_all
    .groupby(["dataset", "mode"])[["bert_f1", "bert_f1_clean", "bert_p", "bert_r", "sem_sim", "sem_sim_clean", "nvlm_sim"]]
    .agg(["mean", "std"])
    .round(3)
)

b2t_delta_summary = (
    b2t_all
    .assign(
        bert_f1_clean_delta=lambda df: df["bert_f1_clean"] - df["bert_f1"],
        sem_sim_clean_delta=lambda df: df["sem_sim_clean"] - df["sem_sim"],
    )
    .groupby(["dataset", "mode"])[["bert_f1_clean_delta", "sem_sim_clean_delta"]]
    .agg(["mean", "std"])
    .round(3)
)

print("=== Brain-to-Text Summary (raw vs cleaned references, mean ± std) ===")
display(summary_b2t)
print("\n=== Cleaned-reference metric deltas (clean - raw, mean ± std) ===")
display(b2t_delta_summary)


### A.5 Recall@1 — Retrieval Rank

Given all N generated texts and N brain images in a dataset × mode group,
**Recall@1** measures the fraction of generated texts whose correct brain image
ranks #1 by `nvlm_sim` over all other brain images.

| Score | Interpretation |
|---|---|
| `1/N` | Chance level (random assignment) |
| `> 2/N` | Better than chance |
| `1.0` | Every generated text uniquely identifies its source brain |

In [ ]:
def recall_at_1(df: pd.DataFrame) -> float:
    """
    Given N (brain, generated_text) pairs, compute the fraction where
    the brain's own generated text scores highest among all texts.
    Chance level = 1/N.
    """
    brain_embs = torch.stack([e.squeeze(0) for e in df["_brain_query_emb"].tolist()])
    nvlm._ensure_projection_heads()
    with torch.no_grad():
        texts    = df["generated"].tolist()
        raw_embs = torch.cat([nvlm._encode_text(t) for t in texts], dim=0)
        z_texts  = nvlm._proj_head_text_infonce(raw_embs.to(nvlm.device))
        z_texts  = F.normalize(z_texts, dim=-1).cpu()
    z_brains   = F.normalize(brain_embs, dim=-1)
    sim_matrix = z_brains @ z_texts.T
    ranks      = (sim_matrix > sim_matrix.diag().unsqueeze(1)).sum(dim=1)
    return float((ranks == 0).float().mean())


# ── Brain query embeddings (fast — no LLM call) ──────────────────────────
print("Computing Recall@1 ...")

brain_emb_cache = {}

def _get_brain_query_emb(name, latent):
    if name not in brain_emb_cache:
        r = nvlm.brain(latent).to_text(datasets=B2T_DATASETS)
        brain_emb_cache[name] = r.query_embeddings
    return brain_emb_cache[name]

_all_latents = {}
for k, d in networks_data.items():
    _all_latents[k] = d["latent"]
for d in pubmed_data_b2t:
    _all_latents[str(d["pmid"])] = d["latent"]
for d in neurovault_data_b2t:
    _all_latents[str(d["doi"])]  = d["latent"]

b2t_all["_brain_query_emb"] = b2t_all["name"].map(
    lambda n: _get_brain_query_emb(n, _all_latents[n])
)

# ── Recall@1 per dataset × mode ───────────────────────────────────────────
r1_rows = []
for (ds, mode), grp in b2t_all.groupby(["dataset", "mode"]):
    grp   = grp.reset_index(drop=True)
    n     = len(grp)
    r1    = recall_at_1(grp)
    chance = round(1 / n, 3)
    r1_rows.append({"dataset": ds, "mode": mode,
                    "N": n, "chance": chance, "recall@1": round(r1, 3)})

b2t_recall1 = pd.DataFrame(r1_rows).set_index(["dataset", "mode"])

print("\n=== Recall@1 ===")
print("  recall@1=1/N means chance; recall@1=1.0 means perfect")
display(b2t_recall1)

In [ ]:
datasets  = ["networks", "pubmed", "neurovault"]
modes     = ["short", "long"]
metrics   = ["nvlm_sim", "bert_f1", "bert_f1_clean", "sem_sim", "sem_sim_clean"]
m_labels  = ["NeuroVLM Latent Sim", "BERTScore F1", "BERTScore F1 (Clean GT)",
             "Semantic Sim", "Semantic Sim (Clean GT)"]

fig, axes = plt.subplots(1, 5, figsize=(24, 4), sharey=False)
x      = np.arange(len(datasets))
w      = 0.35
colors = {"short": "steelblue", "long": "darkorange"}

for ax, metric, m_label in zip(axes, metrics, m_labels):
    for j, mode in enumerate(modes):
        vals = [
            b2t_all[(b2t_all["dataset"] == ds) & (b2t_all["mode"] == mode)][metric].mean()
            for ds in datasets
        ]
        offset = (j - 0.5) * w
        bars = ax.bar(x + offset, vals, w, label=mode,
                      color=colors[mode], alpha=0.85, edgecolor="white")
        ax.bar_label(bars, fmt="%.3f", padding=2, fontsize=7)

    ax.set_xticks(x)
    ax.set_xticklabels(datasets, rotation=15, ha="right")
    ax.set_title(m_label, fontsize=10)
    ax.set_ylabel("Score")
    ax.set_ylim(0, min(1.05, max(0.1, ax.get_ylim()[1] * 1.25)))
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

axes[0].set_facecolor("#fff7e6")
axes[0].text(0.02, 0.95, "domain-specific", transform=axes[0].transAxes,
             fontsize=8, va="top", bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="orange", lw=0.8))

fig.suptitle("Brain-to-Text Generation Quality: Raw vs Cleaned References",
             fontsize=13, fontweight="bold")
fig.tight_layout()
plt.savefig("b2t_metrics_v2.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# NeuroVLM latent sim vs BERTScore F1 scatter — the domain-specific vs general-purpose comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ds_colors  = {"networks": "C0", "pubmed": "C1", "neurovault": "C2"}
ds_markers = {"short": "o", "long": "s"}

for ax, mode in zip(axes, modes):
    sub = b2t_all[b2t_all["mode"] == mode]
    for ds in datasets:
        rows = sub[sub["dataset"] == ds]
        ax.scatter(rows["bert_f1"], rows["nvlm_sim"],
                   s=40, alpha=0.5, color=ds_colors[ds], label=ds,
                   marker=ds_markers[mode])
        ax.scatter(rows["bert_f1"].mean(), rows["nvlm_sim"].mean(),
                   s=180, color=ds_colors[ds], marker="*", zorder=5,
                   edgecolors="black", linewidths=0.5)

    ax.set_xlim(0, 1.05)
    ax.set_ylim(-0.1, 1.05)
    ax.axhline(0, color="grey", lw=0.5, ls="--")
    ax.set_xlabel("BERTScore F1")
    ax.set_ylabel("NeuroVLM Latent Similarity")
    ax.set_title(f"B2T: BERTScore F1 vs NeuroVLM Sim — {mode} mode (★ = dataset mean)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)

fig.tight_layout()
plt.savefig("b2t_bertscore_vs_nvlm.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Part B — Text-to-Brain (Pearson r, Spearman ρ, Dice pct-90, Spin-test p-value)

For each text input we:
1. Generate a predicted brain activation map via `nvlm.text(input).to_brain(head="mse")`.
2. Decode the ground-truth latent through the same autoencoder decoder.
3. Compare with spatial metrics aligned to the NCT reporting style.

**Metrics:**
- **Percentile Dice** (pct=90): thresholds both maps at their own 90th-percentile activation.
- **Spin-test p-value**: projects MNI152 volume maps to fsaverage6/41k using `neuromaps.transforms.mni152_to_fsaverage`, then uses `brainspace.null_models.spin.SpinPermutations` for 1000 rotations. This matches the NCT-style surface-spin significance workflow without reimplementing Wu et al. (2018) registrations.
- **Pearson r / Spearman ρ**: secondary diagnostics for intensity and rank alignment.

PSNR and Top-k overlap are intentionally removed from the primary T2B table.


In [ ]:
def run_t2b(name, text_input, true_latent):
    try:
        # --- Predicted brain map ------------------------------------------
        gen_result  = nvlm.text(text_input).to_brain(head="mse")
        nifti_pred  = gen_result.to_nifti(index=0)
        brain_pred  = masker.transform(nifti_pred).ravel().astype("float32")

        # --- Ground-truth map via same decoder ----------------------------
        dec        = gen_result.decoder
        dec_device = next(dec.parameters()).device
        lat        = true_latent
        if lat.dim() == 1:
            lat = lat.unsqueeze(0)
        with torch.no_grad():
            brain_true = (
                torch.sigmoid(dec(lat.to(dec_device)))
                .squeeze(0).cpu().numpy()
            )
        nifti_true = masker.inverse_transform(brain_true.reshape(1, -1))

        # --- Metrics ------------------------------------------------------
        pearson_r = float(pearson_correlation(brain_true, brain_pred))
        spearman_rho, _ = spearmanr(brain_true.ravel(), brain_pred.ravel())
        dice_p90 = dice_percentile(brain_pred, brain_true, pct=DICE_PCT)
        dice_method = "volume_masker_percentile_fallback"

        pred_lh = pred_rh = true_lh = true_rh = None
        if SPIN_USE_NEUROMAPS:
            try:
                pred_lh, pred_rh = mni152_to_fsaverage_arrays(
                    nifti_pred,
                    density=SPIN_FSAVERAGE_DENSITY,
                    method=SPIN_TRANSFORM_METHOD,
                )
                true_lh, true_rh = mni152_to_fsaverage_arrays(
                    nifti_true,
                    density=SPIN_FSAVERAGE_DENSITY,
                    method=SPIN_TRANSFORM_METHOD,
                )
                nct = nct_dice_spin_test_surface(
                    pred_lh,
                    pred_rh,
                    true_lh,
                    true_rh,
                    pct=DICE_PCT,
                    n_perm=SPIN_TEST_N_PERM,
                    random_state=SPIN_TEST_RANDOM_STATE,
                    density=SPIN_FSAVERAGE_DENSITY,
                )
                dice_p90 = nct.dice_pct
                dice_method = "surface_fsaverage_percentile_nct"
                spin_p_value = nct.spin_p_value
                spin_method = nct.spin_method
                spin_significant = nct.spin_significant
            except ImportError as e:
                spin_p_value = np.nan
                spin_method = f"not_run_missing_dependency:{e.name}"
                spin_significant = False
            except Exception as e:
                spin_p_value = np.nan
                spin_method = f"not_run_nct_error:{type(e).__name__}: {e}"
                spin_significant = False
        else:
            spin_p_value = np.nan
            spin_method = "disabled"
            spin_significant = False

        return {
            "name":         name,
            "text_input":   text_input[:80],
            "pearson_r":    pearson_r,
            "spearman_rho": float(spearman_rho),
            "dice_pct90":   float(dice_p90),
            "dice_method":  dice_method,
            "spin_p_value": float(spin_p_value) if np.isfinite(spin_p_value) else np.nan,
            "spin_significant": bool(spin_significant),
            "spin_method": spin_method,
            "_brain_pred": brain_pred,
            "_brain_true": brain_true,
            "_pred_lh": pred_lh,
            "_pred_rh": pred_rh,
            "_true_lh": true_lh,
            "_true_rh": true_rh,
        }

    except Exception as e:
        print(f"  [T2B error] {name}: {e}")
        return None


### B.1 Networks

Text input = long network description.  
Ground truth = Du atlas network latent.

In [ ]:
t2b_networks_records = []

for net_name, d in tqdm(networks_data.items(), desc="Networks T2B"):
    rec = run_t2b(
        name        = net_name,
        text_input  = d["long_gt"],
        true_latent = d["latent"],
    )
    if rec is not None:
        t2b_networks_records.append(rec)

t2b_net_df = pd.DataFrame(t2b_networks_records).set_index("name")

display_cols = ["pearson_r", "spearman_rho", "dice_pct90", "dice_method", "spin_p_value", "spin_significant", "spin_method"]
t2b_net_df[display_cols].round(3)


### B.2 PubMed

Text input = title + " [SEP] " + abstract.  
Ground truth = pre-encoded PubMed brain image latent.

In [ ]:
t2b_pubmed_records = []

for d in tqdm(pubmed_data_t2b, desc="PubMed T2B"):
    text_in = d["short_gt"] + " [SEP] " + d["long_gt"]
    rec = run_t2b(
        name        = str(d["pmid"]),
        text_input  = text_in,
        true_latent = d["latent"],
    )
    if rec is not None:
        t2b_pubmed_records.append(rec)

t2b_pubmed_df = pd.DataFrame(t2b_pubmed_records)

print(f"PubMed T2B — {len(t2b_pubmed_df)} samples evaluated")
t2b_pubmed_df[["pearson_r", "spearman_rho", "dice_pct90", "spin_p_value"]].describe().round(3)


### B.3 NeuroVault

Text input = title + " [SEP] " + abstract.  
Ground truth = pre-encoded NeuroVault brain image latent.

In [ ]:
t2b_nv_records = []

for d in tqdm(neurovault_data_t2b, desc="NeuroVault T2B"):
    text_in = d["short_gt"] + " [SEP] " + d["long_gt"]
    rec = run_t2b(
        name        = str(d["doi"]),
        text_input  = text_in,
        true_latent = d["latent"],
    )
    if rec is not None:
        t2b_nv_records.append(rec)

t2b_nv_df = pd.DataFrame(t2b_nv_records)

print(f"NeuroVault T2B — {len(t2b_nv_df)} samples evaluated")
t2b_nv_df[["pearson_r", "spearman_rho", "dice_pct90", "spin_p_value"]].describe().round(3)


### B.4 Text-to-Brain Summary

In [ ]:
t2b_net_df_flat = t2b_net_df.reset_index().rename(columns={"name": "sample"})
t2b_net_df_flat["dataset"] = "networks"
t2b_pubmed_df["dataset"]   = "pubmed"
t2b_nv_df["dataset"]       = "neurovault"

t2b_all = pd.concat(
    [t2b_net_df_flat, t2b_pubmed_df, t2b_nv_df],
    ignore_index=True
)

t2b_metric_cols = ["pearson_r", "spearman_rho", "dice_pct90", "spin_p_value"]

summary_t2b = (
    t2b_all
    .groupby("dataset")[t2b_metric_cols]
    .agg(["mean", "std"])
    .round(3)
)

spin_sig_summary = (
    t2b_all
    .groupby("dataset")["spin_significant"]
    .sum()
    .rename("n_spin_p_lt_0_05")
)

print("=== Text-to-Brain Summary (mean ± std) ===")
display(summary_t2b)
print("\n=== Spin-test significant samples (p < 0.05) ===")
display(spin_sig_summary)
print("\nSpin method counts:")
display(t2b_all.groupby(["dataset", "spin_method"]).size().rename("n"))


### B.5 Dice Threshold Sensitivity

Compare top-activation overlap across multiple percentile thresholds. `pct=80` compares the top 20% of each map, `pct=90` compares the top 10%, and `pct=95` compares the top 5%. Spin p-values are recomputed at each threshold when fsaverage surface maps are available.


In [ ]:
def _sensitivity_rows_for_df(df: pd.DataFrame, dataset: str, pcts=DICE_SENSITIVITY_PCTS):
    rows = []
    for _, row in tqdm(df.reset_index().iterrows(), total=len(df), desc=f"{dataset} sensitivity"):
        sample = row.get("name", row.get("sample", row.get("index", "?")))
        has_surface = all(row.get(k) is not None for k in ["_pred_lh", "_pred_rh", "_true_lh", "_true_rh"])
        for pct in pcts:
            if has_surface:
                try:
                    nct = nct_dice_spin_test_surface(
                        row["_pred_lh"],
                        row["_pred_rh"],
                        row["_true_lh"],
                        row["_true_rh"],
                        pct=pct,
                        n_perm=SPIN_TEST_N_PERM,
                        random_state=SPIN_TEST_RANDOM_STATE,
                        density=SPIN_FSAVERAGE_DENSITY,
                    )
                    dice_val = nct.dice_pct
                    spin_p = nct.spin_p_value
                    spin_sig = nct.spin_significant
                    method = "surface_fsaverage_percentile_nct"
                    spin_method = nct.spin_method
                except Exception as e:
                    dice_val = dice_percentile(row["_brain_pred"], row["_brain_true"], pct=pct)
                    spin_p = np.nan
                    spin_sig = False
                    method = "volume_masker_percentile_fallback"
                    spin_method = f"not_run_nct_error:{type(e).__name__}: {e}"
            else:
                dice_val = dice_percentile(row["_brain_pred"], row["_brain_true"], pct=pct)
                spin_p = np.nan
                spin_sig = False
                method = "volume_masker_percentile_fallback"
                spin_method = row.get("spin_method", "not_run_no_surface_maps")

            rows.append({
                "dataset": dataset,
                "sample": sample,
                "pct": pct,
                "top_fraction": (100 - pct) / 100,
                "dice": float(dice_val),
                "spin_p_value": float(spin_p) if np.isfinite(spin_p) else np.nan,
                "spin_significant": bool(spin_sig),
                "dice_method": method,
                "spin_method": spin_method,
            })
    return rows


t2b_sensitivity_records = []
t2b_sensitivity_records.extend(_sensitivity_rows_for_df(t2b_net_df.reset_index(), "networks"))
t2b_sensitivity_records.extend(_sensitivity_rows_for_df(t2b_pubmed_df, "pubmed"))
t2b_sensitivity_records.extend(_sensitivity_rows_for_df(t2b_nv_df, "neurovault"))

t2b_sensitivity_df = pd.DataFrame(t2b_sensitivity_records)

t2b_sensitivity_summary = (
    t2b_sensitivity_df
    .groupby(["dataset", "pct"])
    .agg(
        dice_mean=("dice", "mean"),
        dice_std=("dice", "std"),
        spin_p_median=("spin_p_value", "median"),
        spin_p_lt_0_05=("spin_significant", "mean"),
        n=("sample", "count"),
    )
    .round(3)
)

print("=== T2B Dice / Spin Sensitivity by Threshold ===")
display(t2b_sensitivity_summary)
print("\nMethod counts:")
display(t2b_sensitivity_df.groupby(["dataset", "pct", "dice_method"]).size().rename("n"))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

summary_plot = t2b_sensitivity_df.groupby(["dataset", "pct"]).agg(
    dice_mean=("dice", "mean"),
    dice_sem=("dice", lambda x: x.std() / np.sqrt(len(x))),
    sig_rate=("spin_significant", "mean"),
).reset_index()

for ds, c in zip(["networks", "pubmed", "neurovault"], ["C0", "C1", "C2"]):
    sub = summary_plot[summary_plot["dataset"] == ds]
    axes[0].errorbar(sub["pct"], sub["dice_mean"], yerr=sub["dice_sem"], marker="o", color=c, label=ds)
    axes[1].plot(sub["pct"], sub["sig_rate"], marker="o", color=c, label=ds)

axes[0].set_xlabel("Percentile threshold")
axes[0].set_ylabel("Mean Dice")
axes[0].set_title("T2B Dice Sensitivity")
axes[0].grid(alpha=0.25)
axes[0].legend()

axes[1].set_xlabel("Percentile threshold")
axes[1].set_ylabel("Fraction p < 0.05")
axes[1].set_ylim(-0.05, 1.05)
axes[1].set_title("Spin-Test Significance Rate")
axes[1].grid(alpha=0.25)
axes[1].legend()

fig.tight_layout()
plt.savefig("t2b_dice_threshold_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
ds_order   = ["networks", "pubmed", "neurovault"]
t2b_cols   = ["pearson_r", "spearman_rho", "dice_pct90", "spin_p_value"]
t2b_labels = ["Pearson r", "Spearman ρ", f"Dice (pct={DICE_PCT})", "Spin-test p-value"]
ds_colors  = ["C0", "C1", "C2"]

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=False)
x = np.arange(len(ds_order))

for ax, metric, m_label in zip(axes, t2b_cols, t2b_labels):
    means = [t2b_all[t2b_all["dataset"] == ds][metric].mean() for ds in ds_order]
    stds  = [t2b_all[t2b_all["dataset"] == ds][metric].std()  for ds in ds_order]
    bars  = ax.bar(x, means, color=ds_colors, alpha=0.85, edgecolor="white",
                   yerr=stds, capsize=4, error_kw={"linewidth": 1})
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(ds_order, rotation=15, ha="right")
    ax.set_title(m_label)
    ax.set_ylabel("Score")
    ax.grid(axis="y", alpha=0.3)
    if metric == "spin_p_value":
        ax.axhline(0.05, color="crimson", linestyle="--", linewidth=1)

fig.suptitle("Text-to-Brain Generation Quality by Dataset",
             fontsize=13, fontweight="bold")
fig.tight_layout()
plt.savefig("t2b_metrics_v2.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Pearson r vs Spearman ρ scatter — coloured by dataset
fig, ax = plt.subplots(figsize=(7, 5))

for ds, c in zip(ds_order, ds_colors):
    sub = t2b_all[t2b_all["dataset"] == ds]
    ax.scatter(sub["pearson_r"], sub["spearman_rho"],
               s=40, alpha=0.55, color=c, label=ds)
    ax.scatter(sub["pearson_r"].mean(), sub["spearman_rho"].mean(),
               s=200, color=c, marker="*", zorder=5,
               edgecolors="black", linewidths=0.5)

lim_min = min(t2b_all[["pearson_r", "spearman_rho"]].min()) - 0.05
lim_max = max(t2b_all[["pearson_r", "spearman_rho"]].max()) + 0.05
ax.plot([lim_min, lim_max], [lim_min, lim_max], "--", color="grey", lw=0.8, alpha=0.5)

ax.set_xlabel("Pearson r")
ax.set_ylabel("Spearman ρ")
ax.set_title("Text-to-Brain: Pearson r vs Spearman ρ (★ = dataset mean)")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
plt.savefig("t2b_pearson_vs_spearman.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Full Evaluation Summary

In [ ]:
print("\n" + "="*65)
print("BRAIN-TO-TEXT  (mean across samples)")
print("="*65)
print(
    b2t_all
    .groupby(["dataset", "mode"])[["nvlm_sim", "bert_f1", "bert_f1_clean", "sem_sim", "sem_sim_clean"]]
    .mean()
    .round(4)
    .to_string()
)

print("\nB2T cleaned-reference deltas (clean - raw):")
print(
    b2t_all
    .assign(
        bert_f1_clean_delta=lambda df: df["bert_f1_clean"] - df["bert_f1"],
        sem_sim_clean_delta=lambda df: df["sem_sim_clean"] - df["sem_sim"],
    )
    .groupby(["dataset", "mode"])[["bert_f1_clean_delta", "sem_sim_clean_delta"]]
    .mean()
    .round(4)
    .to_string()
)

print("\n" + "="*65)
print("TEXT-TO-BRAIN  (spatial metrics mean across samples)")
print("="*65)
print(
    t2b_all
    .groupby("dataset")[["pearson_r", "spearman_rho", "dice_pct90", "spin_p_value"]]
    .mean()
    .round(4)
    .to_string()
)

print("\nSpin-test significant counts (p < 0.05):")
print(t2b_all.groupby("dataset")["spin_significant"].sum().to_string())
print("\nSpin method counts:")
print(t2b_all.groupby(["dataset", "spin_method"]).size().to_string())


if "t2b_sensitivity_summary" in globals():
    print("\n" + "="*65)
    print("T2B DICE THRESHOLD SENSITIVITY")
    print("="*65)
    print(t2b_sensitivity_summary.to_string())
